In [7]:
import numpy as np
from PIL import Image
from pathlib import Path
import time
import openflexure_microscope_client as ofm_client
import cv2
import csv
import psutil
import threading
import tracemalloc
import os

MICROSCOPE_IP = ("10.150.79.160")

OUTPUT_BASE_DIR = Path.home() / "Desktop"
OUTPUT_FOLDER_NAME = "single_image_autofocus2807.1.cancer"
output_dir = OUTPUT_BASE_DIR / OUTPUT_FOLDER_NAME
output_dir.mkdir(parents=True, exist_ok=True)

# ---------------- CPU MONITOR ----------------
cpu_usage_list = []
monitoring = False

def monitor_cpu():
    process = psutil.Process()
    while monitoring:
        cpu = process.cpu_percent(interval=0.1)
        cpu_usage_list.append(cpu)

# ---------------- FOCUS METRIC ----------------
def compute_laplacian_variance(image_array):
    if len(image_array.shape) == 3 and image_array.shape[2] == 3:
        gray = cv2.cvtColor(image_array, cv2.COLOR_RGB2GRAY)
    else:
        gray = image_array
    return cv2.Laplacian(gray, cv2.CV_64F).var()

# ---------------- CONNECT ----------------
microscope = ofm_client.MicroscopeClient(MICROSCOPE_IP)
print(f"✅ Connected to microscope at {MICROSCOPE_IP}")

# Confirm Z start
current_z = microscope.position['z']
user_input = input(f"Current Z position is {current_z}. Set start Z? (y/n): ").strip().lower()
if user_input != 'y':
    print("⚠ Set Z manually and rerun.")
    exit()

print("✔ Starting autofocus...\n")

# ---------------- START RAM TRACKING ----------------
process_mem = psutil.Process(os.getpid())
os_ram_start = process_mem.memory_info().rss
tracemalloc.start()

# ---------------- INITIAL STATE ----------------
iteration_start_time = time.time()

z_before = microscope.position['z']
print(f"🔹 Starting Z = {z_before}")

# Capture initial image
initial_image = microscope.grab_image()
initial_array = np.array(initial_image)
initial_vol = compute_laplacian_variance(initial_array)

# Save initial image
initial_path = output_dir / "before_autofocus1.png"
Image.fromarray(initial_array).save(initial_path)

print(f"Initial VoL: {initial_vol:.2f}")

# ---------------- START CPU TRACKING ----------------
monitoring = True
cpu_thread = threading.Thread(target=monitor_cpu)
cpu_thread.start()

# ---------------- AUTOFOCUS ----------------
af_start = time.time()
microscope.autofocus()
af_end = time.time()
af_time = af_end - af_start

# ---------------- FINAL IMAGE ----------------
final_image = microscope.grab_image()
final_array = np.array(final_image)
final_vol = compute_laplacian_variance(final_array)

# Stop CPU tracking
monitoring = False
cpu_thread.join()

avg_cpu = np.mean(cpu_usage_list) if cpu_usage_list else 0

# ---------------- STOP RAM TRACKING ----------------
current_py_mem, peak_py_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

os_ram_end = process_mem.memory_info().rss
os_ram_diff = os_ram_end - os_ram_start

# Convert RAM to MB for easy reading
peak_py_mem_mb = peak_py_mem / (1024 * 1024)
os_ram_diff_mb = os_ram_diff / (1024 * 1024)

# Save final image
final_path = output_dir / "after_autofocus15.png"
Image.fromarray(final_array).save(final_path)

# ---------------- FINAL STATE ----------------
z_after = microscope.position['z']
total_time = time.time() - iteration_start_time

# ---------------- PRINT SUMMARY ----------------
print("\nAutofocus Summary:")
print("──────────────────────────────")
print(f"Z before:              {z_before:.4f}")
print(f"Z after:               {z_after:.4f}")
print(f"Initial VoL:           {initial_vol:.2f}")
print(f"Final VoL:             {final_vol:.2f}")
print(f"Autofocus time:        {af_time:.2f} s")
print(f"Total time:            {total_time:.2f} s")
print("──────────────────────────────")
print("Computational Efficiency:")
print(f"Avg CPU (process):     {avg_cpu:.2f} %")
print(f"Peak Python RAM:       {peak_py_mem_mb:.2f} MB")
print(f"Net OS RAM Footprint:  {os_ram_diff_mb:.2f} MB")
print("──────────────────────────────")

# ---------------- SAVE CSV ----------------
csv_file = output_dir / "results.csv"

file_exists = csv_file.exists()

with open(csv_file, mode='a', newline='') as file:
    writer = csv.writer(file)

    if not file_exists:
        writer.writerow([
            "Z_before", "Z_after",
            "Initial_VoL", "Final_VoL",
            "Autofocus_Time_s", "Total_Time_s",
            "CPU_Usage_percent", "Peak_Py_Mem_MB", "Net_OS_RAM_MB"
        ])

    writer.writerow([
        z_before, z_after,
        initial_vol, final_vol,
        af_time, total_time,
        avg_cpu, peak_py_mem_mb, os_ram_diff_mb
    ])

print(f"\nResults saved to: {csv_file}")
print("Images saved successfully.")
print("✅ Script finished.")

✅ Connected to microscope at 10.150.79.160


Current Z position is -1024. Set start Z? (y/n):  Y


✔ Starting autofocus...

🔹 Starting Z = -1024
Initial VoL: 6.72

Autofocus Summary:
──────────────────────────────
Z before:              -1024.0000
Z after:               -1999.0000
Initial VoL:           6.72
Final VoL:             11.87
Autofocus time:        7.17 s
Total time:            14.51 s
──────────────────────────────
Computational Efficiency:
Avg CPU (process):     1.56 %
Peak Python RAM:       6.96 MB
Net OS RAM Footprint:  2.57 MB
──────────────────────────────

Results saved to: C:\Users\Administrator\Desktop\single_image_autofocus2807.1.cancer\results.csv
Images saved successfully.
✅ Script finished.
